# Representasi TF-IDF dan Reduksi Dimensi Data Berita

Setelah data berita berhasil dikumpulkan dan melalui tahap preprocessing, data teks perlu diubah menjadi bentuk numerik agar dapat diolah lebih lanjut. Pada tahap ini, teks berita direpresentasikan menggunakan TF-IDF (Term Frequency–Inverse Document Frequency) untuk memberikan bobot pada setiap kata berdasarkan tingkat kepentingannya dalam dokumen.

Data hasil TF-IDF memiliki jumlah fitur yang cukup banyak karena setiap kata yang terdapat dalam vocabulary menjadi sebuah fitur. Oleh karena itu, dilakukan reduksi dimensi menggunakan PCA (Principal Component Analysis) untuk mengurangi jumlah fitur dengan tetap mempertahankan sebagian besar informasi dari data.

Tahapan yang dilakukan meliputi pembentukan vocabulary, representasi teks menggunakan TF-IDF, dan reduksi dimensi menggunakan PCA.

### Install Library NLTK

In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('words')

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package words to /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [2]:
import pandas as pd

df = pd.read_csv("dataset_berita.csv")
print(df.shape)
df.head()

(200, 3)


,id,isi_berita,label
0,1,Hamza Abdelkarim kagumi Erling Haaland. Strike...,sport
1,2,Kontrak Luke Shaw di Manchester United akan ha...,sport
2,3,Erling Haaland bermain bagus di Manchester Cit...,sport
3,4,Pengurus Pusat Kickboxing Indonesia (PP KBI) p...,sport
4,5,"Lina Hisage, atlet tolak peluru asal Wamena, P...",sport


## Preprocessing Data

Data berita yang telah dikumpulkan terlebih dahulu dilakukan preprocessing. Pada tahap cleaning, teks diubah menjadi huruf kecil, URL, emoji, tanda baca, dan angka dihapus, serta spasi yang berlebih dirapikan. Setelah itu, teks diubah menjadi token atau kata-kata menggunakan proses tokenisasi.

In [4]:
import re
import emoji

def cleaning(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['isi_bersih'] = df['isi_berita'].apply(cleaning)
df[['isi_berita', 'isi_bersih']].head()

,isi_berita,isi_bersih
0,Hamza Abdelkarim kagumi Erling Haaland. Strike...,hamza abdelkarim kagumi erling haaland striker...
1,Kontrak Luke Shaw di Manchester United akan ha...,kontrak luke shaw di manchester united akan ha...
2,Erling Haaland bermain bagus di Manchester Cit...,erling haaland bermain bagus di manchester cit...
3,Pengurus Pusat Kickboxing Indonesia (PP KBI) p...,pengurus pusat kickboxing indonesia pp kbi per...
4,"Lina Hisage, atlet tolak peluru asal Wamena, P...",lina hisage atlet tolak peluru asal wamena pap...


In [9]:
from nltk.tokenize import word_tokenize

df['tokens'] = df['isi_bersih'].apply(word_tokenize)
df[['isi_bersih', 'tokens']].head()

,isi_bersih,tokens
0,hamza abdelkarim kagumi erling haaland striker...,"[hamza, abdelkarim, kagumi, erling, haaland, s..."
1,kontrak luke shaw di manchester united akan ha...,"[kontrak, luke, shaw, di, manchester, united, ..."
2,erling haaland bermain bagus di manchester cit...,"[erling, haaland, bermain, bagus, di, manchest..."
3,pengurus pusat kickboxing indonesia pp kbi per...,"[pengurus, pusat, kickboxing, indonesia, pp, k..."
4,lina hisage atlet tolak peluru asal wamena pap...,"[lina, hisage, atlet, tolak, peluru, asal, wam..."


Selanjutnya dilakukan stopword removal untuk menghapus kata-kata yang dianggap kurang memiliki informasi penting. Stopword yang digunakan berasal dari Sastrawi dan ditambahkan beberapa kata secara manual.

In [17]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import words as english_words

stopword_factory = StopWordRemoverFactory()
indo_stopwords = set(stopword_factory.get_stop_words())
factory = StemmerFactory()
stemmer = factory.create_stemmer()
english_vocab = set(w.lower() for w in english_words.words())

tambahan_stopwords = {"kali", "mana", "hal", "tim", "era", "pun", "apa", "dan", "dari", "akan", "yang", "saya"}
indo_stopwords = indo_stopwords.union(tambahan_stopwords)

def kata_asing(tokens):
    asing = []
    for t in tokens:
        if t in indo_stopwords:
            continue
        if len(t) <= 3:
            continue
        stem = stemmer.stem(t)
        if t in english_vocab and stem == t:
            asing.append(t)
    return asing

contoh = df['tokens'].iloc[0]
print(kata_asing(contoh))

['hamza', 'striker', 'barcelona', 'hamza', 'striker', 'barcelona', 'hamza', 'main', 'thierry', 'henry', 'hamza', 'striker', 'tribuna', 'cara', 'hamza', 'flick', 'barcelona', 'sedang']


In [18]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in indo_stopwords]

df['tokens_no_stopword'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stopword']].head()

,tokens,tokens_no_stopword
0,"[hamza, abdelkarim, kagumi, erling, haaland, s...","[hamza, abdelkarim, kagumi, erling, haaland, s..."
1,"[kontrak, luke, shaw, di, manchester, united, ...","[kontrak, luke, shaw, manchester, united, habi..."
2,"[erling, haaland, bermain, bagus, di, manchest...","[erling, haaland, bermain, bagus, manchester, ..."
3,"[pengurus, pusat, kickboxing, indonesia, pp, k...","[pengurus, pusat, kickboxing, indonesia, pp, k..."
4,"[lina, hisage, atlet, tolak, peluru, asal, wam...","[lina, hisage, atlet, tolak, peluru, asal, wam..."


Setelah stopword dihapus, dilakukan stemming menggunakan stemmer Bahasa Indonesia untuk mengubah kata menjadi bentuk dasarnya. Dari proses ini dihasilkan kolom tokens_final yang digunakan pada tahap berikutnya.

In [20]:
def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

hasil_stem = []
total = len(df)

for i, tokens in enumerate(df['tokens_no_stopword'], 1):
    hasil_stem.append(stem_tokens(tokens))
    print(f"\rStemming progress: {i}/{total}", end="", flush=True)

df['tokens_final'] = hasil_stem

print("\nSelesai stemming")
df[['tokens_no_stopword', 'tokens_final']].head()

Stemming progress: 200/200
Selesai stemming


,tokens_no_stopword,tokens_final
0,"[hamza, abdelkarim, kagumi, erling, haaland, s...","[hamza, abdelkarim, kagum, erling, haaland, st..."
1,"[kontrak, luke, shaw, manchester, united, habi...","[kontrak, luke, shaw, manchester, united, habi..."
2,"[erling, haaland, bermain, bagus, manchester, ...","[erling, haaland, main, bagus, manchester, cit..."
3,"[pengurus, pusat, kickboxing, indonesia, pp, k...","[urus, pusat, kickboxing, indonesia, pp, kbi, ..."
4,"[lina, hisage, atlet, tolak, peluru, asal, wam...","[lina, hisage, atlet, tolak, peluru, asal, wam..."


## Pembentukan Vocabulary

Setelah preprocessing selesai, seluruh kata dari tokens_final dikumpulkan dan kata yang sama hanya dihitung satu kali. Hasilnya terdapat 5.335 kata unik yang digunakan sebagai vocabulary.

In [21]:
semua_kata = [kata for tokens in df['tokens_final'] for kata in tokens]
reserved_words = {"id", "label"}

vocabulary = sorted(set(
    kata for kata in semua_kata 
    if kata not in reserved_words
))
print(f"Jumlah kata unik setelah filter: {len(vocabulary)}")
print(vocabulary[:30])

Jumlah kata unik setelah filter: 5335
['a', 'aaa', 'ab', 'abad', 'abadi', 'abai', 'abal', 'abang', 'abdelkarim', 'abdoul', 'abdulelah', 'abdullah', 'abel', 'abet', 'abha', 'abiel', 'absen', 'absorber', 'ac', 'academy', 'acak', 'acara', 'accelerating', 'acceptance', 'access', 'accessories', 'accident', 'accreditation', 'acd', 'aceh']


## Representasi Berita dengan TF-IDF

TF-IDF digunakan untuk memberikan bobot pada setiap kata berdasarkan seberapa penting kata tersebut dalam suatu berita. Pada proses ini terdapat dua perhitungan utama, yaitu TF (Term Frequency) dan IDF (Inverse Document Frequency).

Term Frequency (TF):
$$
TF(t,d) = \frac{f(t,d)}{\sum_{t' \in d} f(t',d)}
$$

Keterangan:
* $t$ = kata yang dihitung
* $d$ = dokumen/berita
* $f(t,d)$ = jumlah kemunculan kata $t$ dalam dokumen $d$
* Penyebut = jumlah seluruh kata dalam dokumen

Inverse Document Frequency (IDF):
$$
IDF(t) = \log\left(\frac{N}{df(t)}\right)
$$

Keterangan:
* $N$ = jumlah seluruh dokumen
* $df(t)$ = jumlah dokumen yang mengandung kata t

Kemudian nilai TF dan IDF dikalikan untuk mendapatkan bobot TF-IDF:
$$TFIDF(t,d)=TF(t,d)×IDF(t)$$

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

df['teks_final'] = df['tokens_final'].apply(lambda tokens: ' '.join(tokens))

vectorizer = CountVectorizer(binary=True, vocabulary=vocabulary)
onehot_matrix = vectorizer.fit_transform(df['teks_final'])

onehot_df = pd.DataFrame(onehot_matrix.toarray(), columns=vectorizer.get_feature_names_out())
onehot_df.insert(0, 'id', df['id'].values)
onehot_df['label'] = df['label'].values

print(onehot_df.shape)
onehot_df.head()

(200, 5337)


,id,a,aaa,ab,abad,abadi,abai,abal,abang,abdelkarim,...,zhegrova,zhejiang,zhestkova,zielinski,zikrak,zona,zubimendi,zulhas,zulkifli,label
0,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,sport
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sport


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(vocabulary=vocabulary)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['teks_final'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.insert(0, 'id', df['id'].values)
tfidf_df['label'] = df['label'].values

print(tfidf_df.shape)
tfidf_df.head()

(200, 5337)


,id,a,aaa,ab,abad,abadi,abai,abal,abang,abdelkarim,...,zhegrova,zhejiang,zhestkova,zielinski,zikrak,zona,zubimendi,zulhas,zulkifli,label
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.329679,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport


Teks yang telah melalui preprocessing kemudian digabungkan kembali menjadi teks menggunakan tokens_final. Selanjutnya, TF-IDF digunakan untuk mengubah setiap berita menjadi representasi numerik berdasarkan bobot kepentingan setiap kata dalam dokumen. Hasil representasi menghasilkan 200 berita dengan 5.335 fitur kata.

In [25]:
contoh_doc = tfidf_df.drop(columns=['id', 'label']).iloc[0]
print(contoh_doc.sort_values(ascending=False).head(10))

haaland       0.448107
hamza         0.412099
abdelkarim    0.329679
aku           0.223688
striker       0.216710
erling        0.192046
aspek         0.176326
nil           0.164840
sungai        0.164840
mau           0.161483
Name: 0, dtype: float64


Pada berita pertama, beberapa kata dengan nilai TF-IDF tertinggi adalah persib, seoul, dan bandung. Hal ini menunjukkan bahwa kata-kata tersebut memiliki bobot yang cukup tinggi pada dokumen tersebut dibandingkan kata lainnya.

In [26]:
tfidf_df.to_csv("tfidf_berita.csv", index=False)
print("Selesai! Tersimpan sebagai tfidf_berita.csv")

Selesai! Tersimpan sebagai tfidf_berita.csv


Hasil TF-IDF kemudian disimpan dalam file tfidf_berita.csv untuk digunakan pada proses selanjutnya.

## Reduksi Dimensi dengan PCA

Data hasil TF-IDF kemudian digunakan sebagai input untuk PCA (Principal Component Analysis). Sebelum PCA, data memiliki ukuran 200 × 5.335, yaitu 200 berita dengan 5.335 fitur.

In [27]:
X = tfidf_df.drop(columns=['id', 'label']).values
y = tfidf_df['label'].values

print("Shape sebelum PCA:", X.shape)

Shape sebelum PCA: (200, 5335)


PCA digunakan untuk mengurangi jumlah fitur dengan mempertahankan informasi utama dari data. Pada notebook, jumlah komponen ditentukan berdasarkan cumulative explained variance sebesar 90%. Hasilnya, dibutuhkan 142 komponen dari 5.211 dimensi awal.

In [28]:
from sklearn.decomposition import PCA
import numpy as np

pca_full = PCA()
pca_full.fit(X)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
print(f"Jumlah komponen untuk menjelaskan 90% variansi: {n_components_90}")
print(f"Total dimensi awal: {X.shape[1]}")

Jumlah komponen untuk menjelaskan 90% variansi: 144
Total dimensi awal: 5335


In [29]:
pca = PCA(n_components=n_components_90)
X_reduced = pca.fit_transform(X)

print("Shape setelah PCA:", X_reduced.shape)

pca_df = pd.DataFrame(X_reduced, columns=[f'PC{i+1}' for i in range(X_reduced.shape[1])])
pca_df.insert(0, 'id', tfidf_df['id'].values)
pca_df['label'] = y

pca_df.head()

Shape setelah PCA: (200, 144)


,id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC136,PC137,PC138,PC139,PC140,PC141,PC142,PC143,PC144,label
0,1,-0.124400,-0.011088,0.033212,-0.116010,-0.113935,-0.049799,0.005385,-0.090367,-0.058656,...,-0.057029,-0.003386,-0.015630,0.028462,0.042292,0.086063,0.095085,-0.003322,0.003741,sport
1,2,-0.137792,-0.019039,0.040574,-0.126531,-0.225088,-0.126963,0.009662,-0.004841,0.031886,...,0.027459,-0.019358,0.032893,0.008513,0.021457,0.052109,-0.023725,0.029643,0.044543,sport
2,3,-0.182379,-0.019882,0.074458,-0.164423,-0.187085,-0.119363,-0.022467,-0.072722,-0.037272,...,0.016650,-0.080298,-0.027779,-0.000927,0.053211,-0.035908,-0.119421,0.003319,-0.028239,sport
3,4,0.010424,0.004845,-0.139064,0.027704,-0.112248,0.141588,0.043361,-0.136916,-0.034576,...,0.035082,0.008981,0.053524,0.024143,0.022006,-0.031419,-0.020536,0.012931,-0.010228,sport
4,5,-0.026787,0.021781,-0.127566,-0.000171,-0.099949,0.118858,0.022723,-0.115745,-0.057507,...,0.010047,0.027413,-0.056307,0.038058,0.042651,-0.058196,0.011198,0.052674,0.026050,sport


Setelah dilakukan PCA, ukuran data berubah menjadi 200 × 144. Setiap fitur hasil reduksi direpresentasikan sebagai PC1, PC2, hingga PC144.

In [30]:
pca_df.to_csv("pca_berita.csv", index=False)
print("Selesai! Tersimpan sebagai pca_berita.csv")

Selesai! Tersimpan sebagai pca_berita.csv


Hasil akhir PCA kemudian disimpan dalam file pca_berita.csv.